### Step 1 - Import Libraries and Setup

In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import requests
import os

# Your specific paths
img_path = r"C:\Users\M.T\Desktop\NUST\2nd Sem\DL\DLTutorials\dog.jpeg"
labels_path = r"C:\Users\M.T\Desktop\NUST\2nd Sem\DL\DLTutorials\imagenet_classes.txt"

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cpu


### Step 2: Downloading the Labels

In [58]:
# Download ImageNet labels from the official repository
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"

try:
    response = requests.get(labels_url)
    with open(labels_path, 'wb') as f:
        f.write(response.content)
    
    # Load them into a list for the code to use
    with open(labels_path, 'r') as f:
        categories = [line.strip() for line in f.readlines()]
    print(f"Successfully downloaded and loaded {len(categories)} labels.")
except Exception as e:
    print(f"Download failed: {e}. Check your internet connection.")

Successfully downloaded and loaded 1000 labels.


### Step 3: Preprocessing Pipeline

In [46]:
# Define the preprocessing transformation
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Step 4: Unified Prediction Function

In [85]:
def evaluate_model(model, model_name, actual_label):
    img = Image.open(img_path).convert('RGB')
    img_t = preprocess(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_t)
        probabilities = F.softmax(output[0], dim=0)
    
    # Get top 5 for the breakdown
    top5_prob, top5_idx = torch.topk(probabilities, 5)
    
    print(f" ARCHITECTURE: {model_name.upper()} ")
    print("-" * 30)

    
    print("TOP 5 PREDICTIONS:")
    for i in range(5):
        name = categories[top5_idx[i].item()]
        score = top5_prob[i].item() * 100
        print(f"{i+1}. {name}: {score:.2f}%")
    
    # Comparison Summary
    predicted_label = categories[top5_idx[0].item()]
    print("-" * 30)
    print(f"ACTUAL LABEL:    {actual_label}")
    print(f"PREDICTED LABEL: {predicted_label}")
    
    if actual_label.lower() in predicted_label.lower():
        print("RESULT: CORRECT")
    else:
        print("RESULT: INCORRECT")

### Step 5: VGG-16 Architecture

In [86]:
vgg16 = models.vgg16(weights='DEFAULT').to(device).eval()
evaluate_model(vgg16, "VGG-16", ground_truth)

 ARCHITECTURE: VGG-16 
------------------------------
TOP 5 PREDICTIONS:
1. golden retriever: 91.92%
2. Sussex spaniel: 1.98%
3. Irish setter: 0.89%
4. Labrador retriever: 0.76%
5. cocker spaniel: 0.61%
------------------------------
ACTUAL LABEL:    Golden Retriever
PREDICTED LABEL: golden retriever
RESULT: CORRECT


### Step 6: ResNet-50 Architecture

In [87]:
resnet50 = models.resnet50(weights='DEFAULT').to(device).eval()
evaluate_model(resnet50, "ResNet-50", ground_truth)

 ARCHITECTURE: RESNET-50 
------------------------------
TOP 5 PREDICTIONS:
1. golden retriever: 44.66%
2. tennis ball: 1.07%
3. flat-coated retriever: 0.99%
4. Irish setter: 0.52%
5. Brittany spaniel: 0.42%
------------------------------
ACTUAL LABEL:    Golden Retriever
PREDICTED LABEL: golden retriever
RESULT: CORRECT


### Task 2: Experiment with more architectures

### AlexNet Architecture

In [88]:
alexnet = models.alexnet(weights='DEFAULT').to(device).eval()
evaluate_model(alexnet, "AlexNet", ground_truth)

 ARCHITECTURE: ALEXNET 
------------------------------
TOP 5 PREDICTIONS:
1. cocker spaniel: 75.00%
2. golden retriever: 23.76%
3. Sussex spaniel: 0.50%
4. tennis ball: 0.38%
5. Irish setter: 0.25%
------------------------------
ACTUAL LABEL:    Golden Retriever
PREDICTED LABEL: cocker spaniel
RESULT: INCORRECT


### ResNet-101 Architecture

In [89]:
resnet101 = models.resnet101(weights='DEFAULT').to(device).eval()
evaluate_model(resnet101, "ResNet-101", ground_truth)

 ARCHITECTURE: RESNET-101 
------------------------------
TOP 5 PREDICTIONS:
1. golden retriever: 76.93%
2. tennis ball: 1.38%
3. Irish setter: 0.77%
4. Brittany spaniel: 0.34%
5. Labrador retriever: 0.31%
------------------------------
ACTUAL LABEL:    Golden Retriever
PREDICTED LABEL: golden retriever
RESULT: CORRECT


### MobileNet_V2 Architecture

In [90]:
mobilenet = models.mobilenet_v2(weights='DEFAULT').to(device).eval()
evaluate_model(mobilenet, "MobileNet_V2", ground_truth)

 ARCHITECTURE: MOBILENET_V2 
------------------------------
TOP 5 PREDICTIONS:
1. golden retriever: 10.70%
2. cocker spaniel: 5.21%
3. Irish setter: 2.97%
4. Sussex spaniel: 1.37%
5. clumber: 0.74%
------------------------------
ACTUAL LABEL:    Golden Retriever
PREDICTED LABEL: golden retriever
RESULT: CORRECT
